TP 2 : étude d'un jeu de donnée avec PySpark
Johan Balduzzi INFO FISE 5A

Dans ce TP, nous allons reproduire l'étude de cas du traitement d'un jeu de donnée en utilisant la librairie PySpark.

On commence par importer les librairies nécessaires à cette étude.

In [2]:
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from pyspark.sql.types import LongType
from pyspark.sql.types import FloatType
from pyspark.rdd import RDD
from pyspark.sql.types import StringType
from pyspark.sql.functions import col
import pyspark.sql.functions as F
import csv
from datetime import datetime
from functools import reduce
import pandas as pd
import matplotlib.pyplot as plt

On fait ensuite les configuration pour utiliser une application Spark.

In [3]:
# local[*]: run Spark in local-mode(parallel computing) with as many working processors as logical cores on your machine
# If we want Spark to run locally with 'k' worker threads, we can specify as "local[k]".
master = "local[*]"
# The `appName` field is a name to be shown on the Spark cluster UI page
app_name = "Big data Analysis of Road Crash Data"
# Setup configuration parameters for Spark
spark_conf = SparkConf().setMaster(master).setAppName(app_name)

on crée la session pyspark

In [4]:
spark = SparkSession.builder.config(conf=spark_conf).getOrCreate()
sc = spark.sparkContext
sc.setLogLevel('ERROR')

26/10/16 09:09:42 WARN Utils: Your hostname, DELL-Johan resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/16 09:09:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/16 09:09:43 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


On vérifie que la session est bien crée

In [5]:
print(spark.version, sc.master, sc.defaultParallelism)

3.5.3 local[*] 12


Un RDD est une collection d'objet immuable, découpée en partitions. Spark répartit ces partitions sur les coeurs et applique les opérations en parallèles. Il existe deux façons de crée un RDD: parallelize() à partir d'une collection Python, et textFile() à partir d'un fichier texte.

In [6]:
data_list = [i for i in range(10)]
rdd = sc.parallelize(data_list)
print('Default partitions: ', rdd.getNumPartitions())

rdd = sc.parallelize(data_list, 5)   # on impose 5 partitions
print('Manual partitions: ', rdd.getNumPartitions())

rdd.collect()

Default partitions:  12
Manual partitions:  5


[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

In [7]:
units_rdd = sc.textFile('./units_data/*.csv')
crash_rdd = sc.textFile('./crash_data/*.csv')

Chaque fichier commence par la même ligne d'en-tête. On garde la première ligne en référence et on supprime les lignes identiques.

In [8]:
units_header = units_rdd.first()
units_rdd = units_rdd.filter(lambda row: row != units_header)
print("Total count of units_rdd: ", units_rdd.count())
print(units_rdd.take(10))

crash_header = crash_rdd.first()
crash_rdd = crash_rdd.filter(lambda row: row != crash_header)
print("Total count of crash_rdd: ", crash_rdd.count())
print(crash_rdd.take(10))

Total count of units_rdd:  272591
['"2012-1-21/08/2019","01",0,,"Pedal Cycle",,"South West","Male","035",,,,,"Straight Ahead","001","5074",,', '"2012-1-21/08/2019","02",0,"SA","Motor Cars - Sedan","2009","North West","Female","050","SA","C ","Full","Unknown","Leaving Private Driveway","001","5089",,', '"2012-2-21/08/2019","01",0,"SA","Motor Cars - Sedan","1999","West","Male","027","SA",,"Provisional 1 ","Not Towing","Straight Ahead","002","5008",,', '"2012-2-21/08/2019","02",0,"SA","Motor Cars - Sedan","1993","West",,,,,,"Not Towing","Parked","000",,,', '"2012-3-21/08/2019","01",0,"SA","Motor Cars - Sedan","XXXX","North East","Male","050","SA",,"Full","Trailer","Straight Ahead","006","5107",,', '"2012-3-21/08/2019","02",0,"SA","Motor Cars - Sedan","2006","North East","Male","044","SA",,"Full","Not Towing","Straight Ahead","003","5092",,', '"2012-3-21/08/2019","03",0,,"Other Inanimate Object",,,,,,,,,,,,,', '"2012-4-21/08/2019","01",0,"SA","Station Wagon","2009","South West","Female","0

Total count of crash_rdd:  127672
['"2012-1-21/08/2019","2 Metropolitan","STEPNEY","5069","CC OF NORWOOD,PAYNEHAM & ST PETERS",2,0,0,0,0,2012,"January","Sunday","04:30 pm","060","Not Divided","Straight road","Level","Driveway or Entrance","Sealed","Dry","Not Raining","Daylight","Right Angle","02","Driver Rider","1: PDO","No Control","","",1330659.71,1671795.87,"13306601671796"', '"2012-2-21/08/2019","2 Metropolitan","PARKSIDE","5063","CITY OF UNLEY",2,0,0,0,0,2012,"January","Sunday","09:10 am","040","Not Divided","Straight road","Level","Not Applicable","Sealed","Dry","Not Raining","Daylight","Hit Parked Vehicle","01","Driver Rider","1: PDO","No Control","","",1329400.16,1668462.66,"13294001668463"', '"2012-3-21/08/2019","2 Metropolitan","SELLICKS BEACH","5174","CITY OF ONKAPARINGA",3,0,0,0,0,2012,"January","Wednesday","11:30 am","100","Not Divided","Straight road","Slope","Not Applicable","Sealed","Dry","Not Raining","Daylight","Other","01","Driver Rider","1: PDO","No Control","","",1

On passe au partitionnement

In [9]:
print("Units Data:")
print(f"Total partitions: {units_rdd.getNumPartitions()}")
print("Crash Data:")
print(f"Total partitions: {crash_rdd.getNumPartitions()}")

Units Data:
Total partitions: 8
Crash Data:
Total partitions: 8


On veut maintenant découper les données autrement, selon nos besoins. L'exemple choisi : une partition pour les véhicules immatriculés en Australie-Méridionale (SA) et une partition pour tous les autres cas.

In [10]:
def parseRecord(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))
    return (array_line[3], [x for i, x in enumerate(array_line) if i != 3])

units_state_rdd = units_rdd.map(parseRecord)

def hash_SA(key):
    if key == 'SA':
        return 0
    else:
        return 1

SA_partitioned_rdd = units_state_rdd.partitionBy(2, hash_SA)

In [11]:
def print_partitions(data):
    if isinstance(data, RDD):
        numPartitions = data.getNumPartitions()
        partitions = data.glom().collect()
    else:
        numPartitions = data.rdd.getNumPartitions()
        partitions = data.rdd.glom().collect()

    print(f"NUMBER OF PARTITIONS: {numPartitions}")
    for index, partition in enumerate(partitions):
        if len(partition) > 0:
            print(f"Partition {index}: {len(partition)} records")

print_partitions(SA_partitioned_rdd)

NUMBER OF PARTITIONS: 2
Partition 0: 218366 records
Partition 1: 54225 records


On cherche à savoir l'âge moyen des hommes et femmes impliqués dans un accident. Il faut écarter les lignes où l'âge ou le sexe est inconnu.

In [12]:
def parseGenderAge(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))
    if (array_line[8] != 'XXX') & (array_line[8] != '') & (array_line[7] != 'Unknown'):
        return (array_line[7], int(array_line[8]))

gender_age_rdd = units_rdd.map(parseGenderAge)
filtered_gender_age_rdd = gender_age_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")
gender_age_avg_rdd = filtered_gender_age_rdd.groupByKey().mapValues(lambda x: round(sum(x) / len(x), 2))
gender_age_avg_rdd.collect()

[('Male', 40.8), ('Female', 40.08)]

On veut maintenant l'état d'immatriculation, l'année et le type des véhicules les plus anciens et les plus récents. On écarte les années inconnues.

In [13]:
def parseYear(line):
    lines = line.split(',')
    array_line = []
    for value in lines:
        array_line.append(str(value).replace('"', ''))
    if (array_line[5] != 'XXXX') & (array_line[5] != ''):
        return int(array_line[5]), [array_line[3], array_line[4]]

year_rdd = units_rdd.map(parseYear)
clean_year_rdd = year_rdd.filter(lambda x: x is not None).filter(lambda x: x != "")

print("Oldest year 5 crashes:\n", clean_year_rdd.sortByKey(ascending=True).take(5))
print("\n")
print("Newest year 5 crashes:\n", clean_year_rdd.sortByKey(ascending=False).take(5))

Oldest year 5 crashes:
 [(1900, ['VIC', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle']), (1900, ['SA', 'Motor Cycle'])]




Newest year 5 crashes:
 [(2019, ['SA', 'Station Wagon']), (2019, ['SA', 'OMNIBUS']), (2019, ['SA', 'Motor Cars - Sedan']), (2019, ['SA', 'Station Wagon']), (2019, ['SA', 'SEMI TRAILER'])]


Un DataFrame est une collection distribuée organisée en colonnes nommées, comme une table SQL avec des optimisations en plus.

In [14]:
df = spark.createDataFrame(
    [(1,'Aaditya','A'),(2,'Chinnavit','C'),(3,'Neha','N'),(4,'Huashun','H'),(5,'Mohammad','M'),
     (10,'Prajwol','P'),(1,'Paras','P'),(1,'Tooba','T'),(3,'David','D'),(4,'Cheng','C'),(9,'Haqqani','H')],
    ['Id', 'Name', 'Initial']
)

df.show(5)
df.printSchema()

+---+---------+-------+
| Id|     Name|Initial|
+---+---------+-------+
|  1|  Aaditya|      A|
|  2|Chinnavit|      C|
|  3|     Neha|      N|
|  4|  Huashun|      H|
|  5| Mohammad|      M|
+---+---------+-------+
only showing top 5 rows

root
 |-- Id: long (nullable = true)
 |-- Name: string (nullable = true)
 |-- Initial: string (nullable = true)



L'autre façon de créer un DataFrame est de lire un fichier avec spark.read.csv().

In [15]:
units_df = spark.read.format('csv')\
            .option('header', True).option('escape', '"')\
            .load('./units_data/*.csv')

crash_df = spark.read.format('csv')\
            .option('header', True).option('escape', '"')\
            .load('./crash_data/*.csv')

On vérifie la création.

In [16]:
print(units_df.count(), len(units_df.columns))
print(crash_df.count(), len(crash_df.columns))
units_df.printSchema()

272591 18
127672 33
root
 |-- REPORT_ID: string (nullable = true)
 |-- Unit No: string (nullable = true)
 |-- No Of Cas: string (nullable = true)
 |-- Veh Reg State: string (nullable = true)
 |-- Unit Type: string (nullable = true)
 |-- Veh Year: string (nullable = true)
 |-- Direction Of Travel: string (nullable = true)
 |-- Sex: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Lic State: string (nullable = true)
 |-- Licence Class: string (nullable = true)
 |-- Licence Type: string (nullable = true)
 |-- Towing: string (nullable = true)
 |-- Unit Movement: string (nullable = true)
 |-- Number Occupants: string (nullable = true)
 |-- Postcode: string (nullable = true)
 |-- Rollover: string (nullable = true)
 |-- Fire: string (nullable = true)



On cherche maintenant les accidents survenus à Adelaide où le nombre total de victime dépasse 3.

In [17]:
crash_df = crash_df.withColumn('Total Cas', F.col('Total Cas').cast(IntegerType()))

adelaide_crash_casualty_df = crash_df.filter(col("Total Cas") > 3).filter(col("Suburb") == 'ADELAIDE')

print(adelaide_crash_casualty_df.count())
adelaide_crash_casualty_df.select('REPORT_ID', 'Suburb', 'Total Cas', 'Year', 'Month', 'Day').show()

6
+--------------------+--------+---------+----+--------+--------+
|           REPORT_ID|  Suburb|Total Cas|Year|   Month|     Day|
+--------------------+--------+---------+----+--------+--------+
|2012-4842-21/08/2019|ADELAIDE|        4|2012|   March|Thursday|
|2012-5769-21/08/2019|ADELAIDE|        4|2012|   March|  Friday|
| 2018-601-17/01/2020|ADELAIDE|        4|2018| January|  Sunday|
|2017-1613-15/08/2019|ADELAIDE|        4|2017|February|Saturday|
|2017-12182-15/08/...|ADELAIDE|        5|2017|December|Saturday|
|2019-10404-8/07/2020|ADELAIDE|        6|2019| October|  Monday|
+--------------------+--------+---------+----+--------+--------+



Ensuite les 10 accidents avec le plus de victimes.

In [18]:
crash_df.sort(col('Total Cas'), ascending=False).show(10)

+--------------------+--------------+-------------+--------+--------------------+-----------+---------+----------+--------+--------+----+--------+---------+--------+----------+-------------+--------------------+--------------+--------------+------------+-------------+------------+--------+-----------+---------+------------+-------------+---------------+------------+--------------+----------+----------+--------------+
|           REPORT_ID|    Stats Area|       Suburb|Postcode|            LGA Name|Total Units|Total Cas|Total Fats|Total SI|Total MI|Year|   Month|      Day|    Time|Area Speed|Position Type|    Horizontal Align|Vertical Align|    Other Feat|Road Surface|Moisture Cond|Weather Cond|DayNight| Crash Type|Unit Resp| Entity Code|CSEF Severity|  Traffic Ctrls|DUI Involved|Drugs Involved|  ACCLOC_X|  ACCLOC_Y|    UNIQUE_LOC|
+--------------------+--------------+-------------+--------+--------------------+-----------+---------+----------+--------+--------+----+--------+---------+--

Puis le nombre de décès par type d'accident.

In [19]:
crash_df.groupBy('Crash Type').agg(F.sum('Total Fats').alias('Total fatalities')).sort(col('Total fatalities'), ascending=False).show()

+--------------------+----------------+
|          Crash Type|Total fatalities|
+--------------------+----------------+
|    Hit Fixed Object|           245.0|
|             Head On|           136.0|
|      Hit Pedestrian|           109.0|
|           Roll Over|            93.0|
|         Right Angle|            82.0|
|            Rear End|            33.0|
|          Side Swipe|            30.0|
|          Right Turn|            26.0|
|  Hit Parked Vehicle|            10.0|
|          Hit Animal|             7.0|
|               Other|             5.0|
|Left Road - Out o...|             3.0|
|  Hit Object on Road|             2.0|
+--------------------+----------------+



On veut, pour chaque commune, le nombre total de victimes des accidents impliquant un véhicule conduit par une personne sans permis. Il faut donc joindre les tables.

On commence par regarder les valeurs possible de Licence type qui montre le type de permis. 

In [20]:
units_df.select('Licence Type').distinct().show(truncate=False)

+--------------+
|Licence Type  |
+--------------+
|Unlicenced    |
|Conditional   |
|Disqualified  |
|Unknown       |
|Learners      |
|Probationary  |
|Full          |
|Provisional 2 |
|Provisional 1 |
|NULL          |
+--------------+



On fait la recherche en cherchant la valeur "Unlicenced".

In [21]:
complete_df = crash_df.join(units_df, crash_df.REPORT_ID == units_df.REPORT_ID, how='inner')

by_suburb_no_lic_no_df = complete_df.filter(col('Licence Type') == 'Unlicenced') \
                                    .groupby('Suburb') \
                                    .agg(F.sum('Total Cas').alias('Total casualty'))

by_suburb_no_lic_no_df.sort(col('Total casualty'), ascending=False).show()

+---------------+--------------+
|         Suburb|Total casualty|
+---------------+--------------+
|       ADELAIDE|            43|
|      SALISBURY|            30|
|       PROSPECT|            27|
|     INGLE FARM|            22|
| SALISBURY EAST|            20|
|  MORPHETT VALE|            20|
|  MURRAY BRIDGE|            19|
|      DRY CREEK|            19|
| NORTH ADELAIDE|            17|
|   MAWSON LAKES|            17|
|        ENFIELD|            16|
|    GEPPS CROSS|            15|
|   REGENCY PARK|            15|
|SALISBURY DOWNS|            15|
|         SEATON|            14|
|PARA HILLS WEST|            14|
| ELIZABETH PARK|            14|
|  MOUNT GAMBIER|            13|
|   BEDFORD PARK|            13|
|   DAVOREN PARK|            13|
+---------------+--------------+
only showing top 20 rows



On filtre ensuite par la gravité des accidents.

In [22]:
severe_df = crash_df.groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('Total accidents'))
severe_df.show()

+-------------+---------------+
|CSEF Severity|Total accidents|
+-------------+---------------+
|     4: Fatal|            722|
|        2: MI|          37300|
|       1: PDO|          84775|
|        3: SI|           4875|
+-------------+---------------+



Il y a plusieurs scénarios : On commence par les accidents où au moins un conducteur est positifs aux drogues. On calcule le nombre d'accidents par gravité, puis la part de chaque niveau.

In [23]:
severe_on_drugs_df = crash_df.filter(col('Drugs Involved') == 'Y') \
                             .groupby('CSEF Severity') \
                             .agg(F.count(crash_df.REPORT_ID).alias('Total accidents'))

total_on_drugs = severe_on_drugs_df.groupBy().sum().collect()[0][0]

def find_severe_percent(s):
    return str(round((int(s) / total_on_drugs) * 100, 2)) + "%"

find_severe_percent_udf = udf(find_severe_percent, StringType())

severe_on_drugs_percent_df = severe_on_drugs_df.withColumn('percent_share', find_severe_percent_udf('Total accidents'))
severe_on_drugs_percent_df.show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|     4: Fatal|            117|        6.31%|
|        2: MI|           1109|       59.82%|
|       1: PDO|            237|       12.78%|
|        3: SI|            391|       21.09%|
+-------------+---------------+-------------+



scénario b) alcool : au moins un conducteur avec un taux d'alcool illégal.

On créer un fonction réutilisable pour tous les scénarios restants.

In [24]:
def severity_share(filtered_df):
    counts = filtered_df.groupby('CSEF Severity').agg(F.count('REPORT_ID').alias('Total accidents'))
    total = counts.groupBy().sum().collect()[0][0]
    pct = udf(lambda s: str(round(int(s) / total * 100, 2)) + "%", StringType())
    return counts.withColumn('percent_share', pct('Total accidents'))

In [25]:
severity_share(crash_df.filter(col('DUI Involved') == 'Y')).show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|     4: Fatal|            136|        3.42%|
|        2: MI|           1243|       31.24%|
|       1: PDO|           2144|       53.88%|
|        3: SI|            456|       11.46%|
+-------------+---------------+-------------+



scénario c) alcool et drogues : les deux à la fois.

In [26]:
severity_share(crash_df.filter((col('DUI Involved') == 'Y') & (col('Drugs Involved') == 'Y'))).show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|        2: MI|            151|       50.67%|
|       1: PDO|             36|       12.08%|
|        3: SI|             72|       24.16%|
|     4: Fatal|             39|       13.09%|
+-------------+---------------+-------------+



scénario d) ni l'un ni l'autre : les deux colonnes sont vides. C'est le scénario de référence.

In [27]:
severity_share(crash_df.filter((col('DUI Involved').isNull()) & (col('Drugs Involved').isNull()))).show()

+-------------+---------------+-------------+
|CSEF Severity|Total accidents|percent_share|
+-------------+---------------+-------------+
|     4: Fatal|            508|        0.42%|
|        2: MI|          35099|       28.74%|
|       1: PDO|          82430|       67.49%|
|        3: SI|           4100|        3.36%|
+-------------+---------------+-------------+



On combine les tableaux.

In [28]:
drugs_positive_df = crash_df.filter(col('Drugs Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Drugs'))
alcohol_positive_df = crash_df.filter(col('DUI Involved') == 'Y').groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Alcohol'))
alcohol_drugs_positive_df = crash_df.filter((col('DUI Involved') == 'Y') & (col('Drugs Involved') == 'Y')).groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On Both'))
drugs_alcohol_negative_df = crash_df.filter((col('DUI Involved').isNull()) & (col('Drugs Involved').isNull())).groupby('CSEF Severity').agg(F.count(crash_df.REPORT_ID).alias('On None'))

In [29]:
all_df = drugs_positive_df.join(alcohol_positive_df, ["CSEF Severity"]) \
                          .join(alcohol_drugs_positive_df, ["CSEF Severity"]) \
                          .join(drugs_alcohol_negative_df, ["CSEF Severity"])
all_df.show()

+-------------+--------+----------+-------+-------+
|CSEF Severity|On Drugs|On Alcohol|On Both|On None|
+-------------+--------+----------+-------+-------+
|     4: Fatal|     117|       136|     39|    508|
|        2: MI|    1109|      1243|    151|  35099|
|       1: PDO|     237|      2144|     36|  82430|
|        3: SI|     391|       456|     72|   4100|
+-------------+--------+----------+-------+-------+



In [32]:
%pip install setuptools

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [30]:
pd_df = all_df.toPandas().sort_values('CSEF Severity').reset_index(drop=True)

cols = ['On Drugs', 'On Alcohol', 'On Both', 'On None']
pd_df_per = pd_df.copy()
pd_df_per[cols] = (pd_df[cols] / pd_df[cols].sum() * 100).round(2)

print(pd_df)
print(pd_df_per)

ModuleNotFoundError: No module named 'distutils'